# Data processing

## Airport workers 
This part downloads the census and jobs data, finds where airport workers live, and computes a population-weighted centrer point for each Bay Area tract.

In [1]:
from src.config import save_log
from src.lodes import load_blocks, load_lodes, load_airports
from src.spatial import airport_workers_by_tract, tract_centroids

### Load data

In [2]:
blocks = load_blocks()
lodes = load_lodes()
airports = load_airports()
print(f"blocks: {len(blocks):,}  lodes: {len(lodes):,}  airports: {len(airports)}")

[blocks] keep 9 counties: 519,723 -> 82,210 (block outside the 9 counties)


[airports] match airport names: 8,110 -> 3 (TIGER landmark is not an SFO, OAK or SJC airport (k2457))
blocks: 82,210  lodes: 16,241,068  airports: 3


### Airport workers by home tract

In [3]:
workers = airport_workers_by_tract(blocks, lodes, airports)
print(f"rows: {len(workers):,}")
workers.groupby("code")[["S000", "SE01", "SE02", "SE03"]].sum()

[blocks] inside airport: 82,210 -> 82 (not on airport land)


[lodes] work at airport: 16,241,068 -> 19,631 (workplace not an airport block)
[airport_workers] home in 9 counties: 6,719 -> 3,768 (4,072 of 22,702 workers live outside the 9 counties)
rows: 3,768


,S000,SE01,SE02,SE03
code,,,,
OAK,3067,272,947,1848
SFO,11646,688,1728,9230
SJC,3917,485,835,2597


### Population-weighted tract centroids

In [4]:
centroids = tract_centroids(blocks)
print(f"tracts: {len(centroids):,}")
centroids.head()

[tracts] drop zero population: 1,772 -> 1,763 (POP20 = 0)
tracts: 1,763


,GEOID,POP20,geometry
0,06001400100,3038,POINT (567733.804 4190483.267)
1,06001400200,2001,POINT (566029.137 4189246.747)
2,06001400300,5504,POINT (565579.859 4188333.843)
3,06001400400,4112,POINT (565340.101 4189261.285)
4,06001400500,3644,POINT (564680.062 4189315.6)


## Airport stops
This part finds the transit stops on airport land, using the same TIGER airport polygons as the airport workers. Drive trips end at one terminal stop per airport, and transit trips at any of them.

In [5]:
from src.transit import airport_stops

In [6]:
stops = airport_stops(airports)
stops.groupby("code").size()

[gtfs_stops] inside airport: 19,802 -> 49 (stop not on airport land)
[gtfs_stops] drop Santa Clara rail: 49 -> 47 (rail platform inside the SJC polygon, still a bus ride from the terminals)


code
OAK     2
SFO    38
SJC     7
dtype: int64

## Transit
This part downloads the 511 regional transit feed, routes every tract to each airport with r5py leaving home at 2 am on a Wednesday and a Sunday, and keeps each tract's earliest airport arrival up to 9 am.

Assumes a 511 API token is present (obtain from https://511.org/open-data/token and run `export KEY_511=your_token`)

In [7]:
import datetime

from src.config import SERVICE_DATE
from src.transit import add_overnight_trips, download_gtfs, earliest_arrival, transit_network, transit_times, transit_trip_times

### Download the 511 feed

In [8]:
download_gtfs()

### Add overnight trips
r5py only runs a day's own trips, so trips from the night before that run past midnight (like SamTrans 397 to SFO) are copied onto each day.

In [9]:
gtfs = add_overnight_trips()

[gtfs_trips] add overnight trips for 2026-10-28: 87,764 -> 88,633 (trip from the night before still running after midnight)


[gtfs_trips] add overnight trips for 2026-11-08: 87,764 -> 88,607 (trip from the night before still running after midnight)


### Build the routing network

In [10]:
network = transit_network(gtfs)

### Test route: downtown Berkeley to SFO
Sanity check. We use the population-weighted centroid of the tract containing Downtown Berkeley BART. In the 511 timetable for Wednesday Oct 28, the Red line leaves Downtwn Berkeley at 12:24pm and arrives at SFO station at 1:22pm, a 58-minute ride. So if a rider leaves home 15 12:15, should be 67 min door to door.

In [11]:
# tract with downtown Berkeley BART
berkeley = centroids[centroids["GEOID"] == "06001422902"]
transit_times(network, berkeley, stops[stops["code"] == "SFO"], datetime.datetime.fromisoformat(f"{SERVICE_DATE} 12:15"))

,GEOID,code,transit_min
0,06001422902,SFO,67


### Earliest arrival

In [12]:
first = earliest_arrival(network, centroids, stops)
# arrival in minutes after midnight (e.g. 5 am = 300)
first.groupby(["day", "code"])["arrival_min"].describe().round(0)

[transit] reaches airport by 9 am: 10,578 -> 7,603 (no transit trip leaving at 2 am reaches the airport by 9 am)


count   mean    std    min    25%    50%    75%    max
day        code                                                         
2026-10-28 OAK   1488.0  359.0   91.0  147.0  267.0  376.0  432.0  537.0
           SFO   1470.0  348.0  101.0  144.0  249.0  375.0  433.0  535.0
           SJC   1412.0  386.0   62.0  145.0  353.0  376.0  416.0  534.0
2026-11-08 OAK   1143.0  383.0  125.0  147.0  267.0  397.0  517.0  537.0
           SFO   1063.0  350.0  130.0  144.0  236.0  309.0  486.0  534.0
           SJC   1027.0  409.0   74.0  145.0  355.0  416.0  484.0  538.0

### Transit trip times
For the transit vs car comparison: the typical trip leaving home 4-5 am (early shift) and 7-8 am (normal commute) on the weekday.

In [13]:
trips = transit_trip_times(network, centroids, stops)
trips.groupby(["window", "code"])["transit_min"].describe().round(0)

count   mean   std   min    25%    50%    75%    max
window code                                                       
4-5 am OAK   1203.0  113.0  37.0  35.0   78.0  111.0  149.0  179.0
       SFO   1241.0  115.0  37.0  24.0   87.0  115.0  146.0  179.0
       SJC   1107.0  117.0  30.0  25.0  104.0  121.0  135.0  179.0
7-8 am OAK   1428.0  104.0  36.0  32.0   73.0  102.0  134.0  179.0
       SFO   1422.0  101.0  38.0  21.0   68.0  103.0  129.0  179.0
       SJC   1281.0  106.0  34.0  16.0   79.0  111.0  128.0  179.0

## Road network
This part uses OSMnx (from class) for street network routing. We keep public roads, run one test route from downtown San Francisco to SFO, and find the drive time from every tract to each airport.

In [15]:
from src.network import drive_times, load_road_network, shortest_route

### Load road network

In [16]:
G = load_road_network()
print(f"nodes: {len(G.nodes):,}  edges: {len(G.edges):,}")

[road_network] public roads only: 1,749,823 -> 1,267,864 (edge is a footpath, parking aisle, driveway, or private/no-access road)


[road_network] largest connected component: 729,478 -> 539,070 (node in one of 176939 small pieces you can't drive both into and out of)


nodes: 539,070  edges: 1,234,786


### Test route: downtown San Francisco to SFO

In [17]:
# downtown San Francisco to the SFO Terminal 2 curb (lon, lat)
route = shortest_route(G, -122.4194, 37.7749, -122.38414, 37.61609)
print(f"route distance: {route.length.sum() / 1000:.1f} km")
route.to_file("data/processed/sf_to_sfo_test_route.gpkg")

endpoint snapped 21 m to the road network


route distance: 20.8 km


### Drive times to each airport

In [18]:
drive = drive_times(G, centroids, stops)
drive.groupby("code")["drive_min"].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
code,,,,,,,,
OAK,1763.0,35.5,16.9,4.8,25.5,33.5,42.0,146.6
SFO,1763.0,36.6,19.4,3.4,23.9,33.3,44.6,150.9
SJC,1763.0,43.2,26.4,3.8,21.6,42.1,54.9,175.9


### Transit vs car

In [19]:
compare = trips.merge(drive, on=["GEOID", "code"])
# how many times longer transit takes than driving
compare["transit_over_drive"] = (compare["transit_min"] / compare["drive_min"]).round(1)
compare.to_csv("data/processed/transit_vs_drive.csv", index=False)
compare.groupby(["window", "code"])[["transit_min", "drive_min", "transit_over_drive"]].median()

transit_min  drive_min  transit_over_drive
window code                                            
4-5 am OAK         111.0  33.450508                 3.9
       SFO         115.0  33.314563                 4.1
       SJC         121.0  42.143108                 3.7
7-8 am OAK         102.0  33.450508                 3.5
       SFO         103.0  33.314563                 3.3
       SJC         111.0  42.143108                 3.3

### Cleaning log

In [20]:
save_log()
import pandas as pd
pd.read_csv("outputs/cleaning_log.csv")

,dataset,step,rows_before,rows_after,reason
0,blocks,keep 9 counties,519723,82210,block outside the 9 counties
1,airports,match airport names,8110,3,"TIGER landmark is not an SFO, OAK or SJC airpo..."
2,blocks,inside airport,82210,82,not on airport land
3,lodes,work at airport,16241068,19631,workplace not an airport block
4,airport_workers,home in 9 counties,6719,3768,"4,072 of 22,702 workers live outside the 9 cou..."
5,tracts,drop zero population,1772,1763,POP20 = 0
6,gtfs_stops,inside airport,19802,49,stop not on airport land
7,gtfs_stops,drop Santa Clara rail,49,47,"rail platform inside the SJC polygon, still a ..."
8,gtfs_trips,add overnight trips for 2026-10-28,87764,88633,trip from the night before still running after...
9,gtfs_trips,add overnight trips for 2026-11-08,87764,88607,trip from the night before still running after...
